In [12]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')

### Parameters

In [13]:
EXPORT_FOLDER = 'GEMLST_MODIS'
TILE_SCALE = 8
SCALE_M = 1000


### Initialize variables: Mask, extraction points and time frame 

In [14]:
date_start = '2012-01-19' # First VIIRS
date_end = '2025-12-31'

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

poi = ee.FeatureCollection("projects/ee-ivanburgov666/assets/randomGR5km_masked_260508")


In [15]:
# Mask data based on quality flags analogously to extraction script from GEM stations. 
def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskViirs(image):
    '''Function to filter VIIRS LST data based on quality flag.'''
    qa = image.select('QC')
    bits01Mask = bitwiseExtract(qa, 0, 1).eq(0); 
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    bit1213Mask = bitwiseExtract(qa, 12, 13).gte(2)
    bit1415Mask = bitwiseExtract(qa, 14, 15).gte(2)
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit1213Mask).And(bit1415Mask)
    return image.updateMask(mask)


In [16]:
# Load VIIRS data, apply quality control and conversion functions

# VIIRS 

def lst_viirs(image):
    'VIIRS band selection and conversion'
    lst = image.select('LST_1KM').subtract(273.15).rename('VIIRS_LST_1KM_C')
    qa = image.select('QC')
    overfly = image.select('View_Time').multiply(0.1).rename('time')
    # Keep only converted LST, QC and overfly-time bands before sampling
    return image.addBands(lst).addBands(qa).addBands(overfly).select(['VIIRS_LST_1KM_C','QC','time'])

VIIRS_Day = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .select(['LST_1KM', 'QC', 'View_Time'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs)
 )

VIIRS_Night = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1N")
    .select(['LST_1KM', 'QC', 'View_Time']) 
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs)
 )

# Print the number of images in imgTerraD
print('Number of images in VIIRS_Day:', VIIRS_Day.size().getInfo())

Number of images in VIIRS_Day: 5057


In [17]:
# Add a date property so malformed night granules can be filtered before sampling.
BAD_VIIRS_NIGHT_DATES = ['2021-08-04', '2022-06-28', '2023-07-27', '2023-07-26', '2024-04-25', '2024-07-16', '2024-07-29']

def add_date_prop(img):
    return img.set('date', ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'))

def filter_bad_viirs_night_granules(collection):
    return (
        collection
        .map(add_date_prop)
        .filter(ee.Filter.inList('date', BAD_VIIRS_NIGHT_DATES).Not())
    )


In [18]:
# def find_bad_viirs_dates(collection, skip_dates=None, test_points=None):
#     """Probe each image with a tiny sampleRegions call and collect failing dates."""
#     probe_points = test_points if test_points is not None else poi_with_id.limit(1)
#     skip_dates = set(skip_dates or [])
#     dates = collection.aggregate_array('date').getInfo()
#     bad_dates = []

#     for date in dates:
#         if date in skip_dates:
#             continue
#         try:
#             image = ee.Image(collection.filter(ee.Filter.eq('date', date)).first())
#             _ = image.sampleRegions(
#                 collection=probe_points,
#                 properties=['object_id', 'class'],
#                 scale=SCALE_M,
#                 tileScale=1,
#                 geometries=False,
#             ).first().getInfo()
#         except Exception as exc:
#             print(f'Bad VIIRS granule {date}: {exc}')
#             bad_dates.append(date)

#     return bad_dates

# # Run the probe on the raw night collection so only the remaining unknown dates are tested.
# VIIRS_Night_for_probe = VIIRS_Night.map(add_date_prop)
# new_bad_viirs_night_dates = find_bad_viirs_dates(
#     VIIRS_Night_for_probe,
#     skip_dates=BAD_VIIRS_NIGHT_DATES,
# )
# print('New bad dates found:', new_bad_viirs_night_dates)


In [19]:
def add_poi_id(feature):
    poi_id = feature.get('object_id')
    cls = feature.get('class')
    return ee.Feature(feature).set({'object_id': poi_id, 'class': cls})

poi_with_id = poi.map(add_poi_id)

def extract_collection_at_poi(collection):
    def sample_image(img):
        date = ee.String(img.get('date'))
        sampled = img.sampleRegions(
            collection=poi_with_id,
            properties=['object_id', 'class'],
            scale=SCALE_M,
            tileScale=TILE_SCALE,
            geometries=False,
        )
        return sampled.map(
            lambda f: ee.Feature(f).set(
                {
                    'object_id': f.get('object_id'),
                    'class': f.get('class'),
                    'date': date,
                    'sample_key': ee.String(f.get('object_id')).cat('_').cat(ee.String(f.get('class'))).cat('_').cat(date),
                }
            )
        )

    return ee.FeatureCollection(collection.map(sample_image).flatten())

VIIRS_Day = VIIRS_Day.map(add_date_prop)
VIIRS_Night = filter_bad_viirs_night_granules(VIIRS_Night)

poiLST_viirsD = extract_collection_at_poi(VIIRS_Day)
poiLST_viirsN = extract_collection_at_poi(VIIRS_Night)

sensors = {
    'VIIRS_Day': poiLST_viirsD,
    #'VIIRS_Night': poiLST_viirsN
}


### Export

In [20]:
def export(sensor_name, sensor_value):
    description = f'GEMLST_{sensor_name}_LST_POI'
    selector_map = {
        'VIIRS_Day': ['object_id','class','date','VIIRS_LST_1KM_C','VIIRS_QC','time'],
        'VIIRS_Night': ['object_id','class','date','VIIRS_LST_1KM_C','VIIRS_QC','time'],
    }
    
    selectors = selector_map[sensor_name]

    task = ee.batch.Export.table.toDrive(
        collection=sensor_value,
        description=description,
        fileFormat='CSV',
        # projection='EPSG:3413', # kept native MODIS projection to accellerate sampling
        folder=EXPORT_FOLDER,
        selectors=selectors,
    )
    task.start()
    print(f'Started export task: {description}')

for sensor_name, sensor_values in sensors.items():
    export(sensor_name, sensor_values)
print('All export tasks started.')


Started export task: GEMLST_VIIRS_Day_LST_POI
All export tasks started.
